# MOSAIC + RTF Encoder → Classifier (normalized + calibrated)

Fixes vs previous run:
- **Global** median/IQR on the 4 base channels (fit train → apply val/test)
- **Global** median/IQR on meta channels (not per-object)
- Val-calibrated thresholds (scores may still be soft)
- Clean train eval metrics (no dropout / sampler noise)
- Scalar logistic baseline for comparison


In [ ]:
import os, sys
if not os.path.isdir("/content/rtf"):
    !git clone https://github.com/applecider-ml/rtf.git /content/rtf
sys.path.insert(0, "/content/rtf/src")

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc, average_precision_score,
    precision_recall_curve, confusion_matrix, roc_auc_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
from collections import Counter
from model import LightCurveCompressor

ALERT_META_KEYS = [
    "sharpnr", "scorr", "diffmaglim", "sky", "sigmapsf",
    "chinr", "rb", "chipsf", "distnr", "magnr", "fwhm",
]
BASE_KEYS = ["log_dt", "log_dt_prev", "logflux", "logflux_err"]
N_BANDS = 3
MAX_LEN = 257
IN_CHANNELS = 18
SEED = 42
EPOCHS = 50
BATCH_SIZE = 128
LR = 2e-4
LR_HEAD = 1e-3
LATENT_DIM = 128
IMP_PER_KN = 5
USE_POS_WEIGHT = False  # sampler already balances; do NOT stack pos_weight

torch.manual_seed(SEED)
np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

PARQUET_TRAIN = "/content/train.parquet"
PARQUET_VAL   = "/content/val.parquet"
PARQUET_TEST  = "/content/test.parquet"


## 1. Global stats (fit on train only)


In [ ]:
def _collect_raw_features(path):
    """Gather unnormalized base + meta values across all objects (for global stats)."""
    df = pd.read_parquet(path)
    buckets = {k: [] for k in BASE_KEYS + ALERT_META_KEYS}
    for oid, g in df.groupby("objectId"):
        g = g.sort_values("jd")
        g = g[g["fid"].isin([1, 2, 3])]
        if len(g) < 3:
            continue
        if len(g) > MAX_LEN:
            g = g.iloc[:MAX_LEN]
        L = len(g)
        jds = g["jd"].values.astype(np.float64)
        mags = g["magpsf"].values.astype(np.float32)
        sigs = g["sigmapsf"].values.astype(np.float32)
        dt = (jds - jds[0]).astype(np.float32)
        dtp = np.zeros(L, np.float32)
        dtp[1:] = np.diff(jds).astype(np.float32)
        buckets["log_dt"].extend(np.log1p(dt).tolist())
        buckets["log_dt_prev"].extend(np.log1p(dtp).tolist())
        buckets["logflux"].extend((-0.4 * mags).tolist())
        buckets["logflux_err"].extend((0.4 * sigs).tolist())
        for key in ALERT_META_KEYS:
            if key in g.columns:
                buckets[key].extend(g[key].fillna(0.0).astype(np.float32).tolist())
            else:
                buckets[key].extend([0.0] * L)
    return buckets


def fit_global_stats(path):
    print(f"Fitting global stats on {path}...")
    buckets = _collect_raw_features(path)
    stats = {}
    for k, vals in buckets.items():
        a = np.asarray(vals, dtype=np.float64)
        if len(a) == 0:
            stats[k] = {"median": 0.0, "iqr": 1.0}
            continue
        med = float(np.median(a))
        iqr = float(max(np.percentile(a, 75) - np.percentile(a, 25), 1e-6))
        stats[k] = {"median": med, "iqr": iqr}
        print(f"  {k:16s} med={med:9.4f} iqr={iqr:9.4f}")
    return stats


def _norm(x, key, stats):
    return (x - stats[key]["median"]) / stats[key]["iqr"]


def load_rtf_tensors(path, stats):
    print(f"Loading {path}...")
    df = pd.read_parquet(path)
    xs, masks, ys, subs, npts = [], [], [], [], []
    for oid, g in df.groupby("objectId"):
        g = g.sort_values("jd")
        g = g[g["fid"].isin([1, 2, 3])]
        if len(g) < 3:
            continue
        if len(g) > MAX_LEN:
            g = g.iloc[:MAX_LEN]
        L = len(g)
        jds = g["jd"].values.astype(np.float64)
        mags = g["magpsf"].values.astype(np.float32)
        sigs = g["sigmapsf"].values.astype(np.float32)
        fids = g["fid"].values.astype(np.int64)

        dt = (jds - jds[0]).astype(np.float32)
        dtp = np.zeros(L, np.float32)
        dtp[1:] = np.diff(jds).astype(np.float32)

        c0 = np.clip(_norm(np.log1p(dt), "log_dt", stats), -10, 10)
        c1 = np.clip(_norm(np.log1p(dtp), "log_dt_prev", stats), -10, 10)
        c2 = np.clip(_norm(-0.4 * mags, "logflux", stats), -10, 10)
        c3 = np.clip(_norm(0.4 * sigs, "logflux_err", stats), -10, 10)
        base = np.column_stack([c0, c1, c2, c3]).astype(np.float32)
        one_hot = np.eye(N_BANDS, dtype=np.float32)[fids - 1]

        meta = np.zeros((L, len(ALERT_META_KEYS)), np.float32)
        for j, key in enumerate(ALERT_META_KEYS):
            if key in g.columns:
                raw = g[key].fillna(0.0).values.astype(np.float32)
            else:
                raw = np.zeros(L, np.float32)
            meta[:, j] = np.clip(_norm(raw, key, stats), -10, 10)

        x = np.concatenate([base, one_hot, meta], axis=1).astype(np.float32)
        if MAX_LEN - L > 0:
            x = np.concatenate([x, np.zeros((MAX_LEN - L, IN_CHANNELS), np.float32)])
        mask = np.zeros(MAX_LEN, dtype=bool)
        mask[L:] = True

        ys.append(int(g["label_class"].iloc[0] == 1) if "label_class" in g.columns else 0)
        subs.append(g["subclass"].iloc[0] if "subclass" in g.columns else "Unknown")
        npts.append(L)
        xs.append(x)
        masks.append(mask)

    X = torch.tensor(np.stack(xs))
    M = torch.tensor(np.stack(masks))
    Y = np.array(ys, dtype=np.int64)
    print(f"  -> {len(Y)} objects  KN={int((Y==1).sum())}  imp={int((Y==0).sum())}")
    return X, M, Y, np.array(subs), np.array(npts)


class DS(Dataset):
    def __init__(self, X, M, Y):
        self.X, self.M, self.Y = X, M, Y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.M[i], int(self.Y[i])


## 2. Model


In [ ]:
class RTFEncoderClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.rtf = LightCurveCompressor(
            mode="ae", latent_dim=LATENT_DIM, in_channels=IN_CHANNELS,
            d_model=256, use_images=False, gp_dim=0, num_classes=0,
        )
        self.head = nn.Sequential(
            nn.Linear(LATENT_DIM, 64), nn.GELU(), nn.Dropout(0.2), nn.Linear(64, 1),
        )

    def forward(self, x, pad_mask):
        return self.head(self.rtf.embed(x, pad_mask)).squeeze(-1)

model = RTFEncoderClassifier().to(device)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 3. Train (clean val + optional clean train metrics)


In [ ]:
stats = fit_global_stats(PARQUET_TRAIN)
Xtr, Mtr, Ytr, _, _ = load_rtf_tensors(PARQUET_TRAIN, stats)
Xva, Mva, Yva, _, _ = load_rtf_tensors(PARQUET_VAL, stats)

n_kn = max(int((Ytr == 1).sum()), 1)
n_imp = max(int((Ytr == 0).sum()), 1)
w = np.zeros(len(Ytr), dtype=np.float64)
w[Ytr == 1] = 1.0
w[Ytr == 0] = (IMP_PER_KN * n_kn) / float(n_imp)
sampler = WeightedRandomSampler(w, num_samples=min(len(Ytr), n_kn * (1 + IMP_PER_KN)), replacement=True)

train_loader = DataLoader(DS(Xtr, Mtr, Ytr), batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
# fixed loaders for clean metrics (no dropout, no resampling)
train_eval_loader = DataLoader(DS(Xtr, Mtr, Ytr), batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
val_loader = DataLoader(DS(Xva, Mva, Yva), batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# Sampler rebalances batches (~1 KN : IMP_PER_KN imp).
# Do not also use raw n_imp/n_kn pos_weight — that double-counts imbalance.
if USE_POS_WEIGHT:
    # Only if you disable the sampler: weight by original class ratio
    pw = torch.tensor([n_imp / float(n_kn)], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    print(f"pos_weight = {pw.item():.3f}  (WARNING: stack with sampler = over-weight KN)")
else:
    criterion = nn.BCEWithLogitsLoss()
    print("BCE plain — balance via WeightedRandomSampler only")

opt = torch.optim.AdamW([
    {"params": model.rtf.parameters(), "lr": LR},
    {"params": model.head.parameters(), "lr": LR_HEAD},
], weight_decay=1e-2)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

hist = {k: [] for k in ["tr_l", "va_l", "tr_a", "va_a", "tr_ap", "va_ap"]}
best_ap, best_state = -1.0, None

def run_epoch(loader, train_mode):
    model.train(train_mode)
    tot, n = 0.0, 0
    ps, ys = [], []
    for x, m, y in loader:
        x, m, y = x.to(device), m.to(device), y.float().to(device)
        if train_mode:
            opt.zero_grad(set_to_none=True)
        logit = model(x, m)
        loss = criterion(logit, y)
        if train_mode:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        tot += loss.item() * x.size(0)
        n += x.size(0)
        ps.append(torch.sigmoid(logit.detach()).cpu())
        ys.append(y.cpu())
    p = torch.cat(ps).numpy()
    yy = torch.cat(ys).numpy().astype(int)
    a = auc(*roc_curve(yy, p)[:2]) if len(np.unique(yy)) > 1 else float("nan")
    ap = average_precision_score(yy, p) if len(np.unique(yy)) > 1 else float("nan")
    return tot / max(n, 1), a, ap

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    # training step (sampler + dropout) — loss only for optimization signal
    tr_l, _, _ = run_epoch(train_loader, True)
    # clean metrics
    tr_l_c, tr_a, tr_ap = run_epoch(train_eval_loader, False)
    va_l, va_a, va_ap = run_epoch(val_loader, False)
    sched.step()
    hist["tr_l"].append(tr_l_c); hist["va_l"].append(va_l)
    hist["tr_a"].append(tr_a); hist["va_a"].append(va_a)
    hist["tr_ap"].append(tr_ap); hist["va_ap"].append(va_ap)
    if va_ap > best_ap:
        best_ap = va_ap
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"Epoch {ep:02d}/{EPOCHS}  loss {tr_l_c:.4f}/{va_l:.4f}  "
          f"AUC {tr_a:.4f}/{va_a:.4f}  AP {tr_ap:.4f}/{va_ap:.4f}  ({time.time()-t0:.1f}s)")

if best_state:
    model.load_state_dict(best_state)
print(f"Best val AP = {best_ap:.4f}")
torch.save({"model": model.state_dict(), "stats": stats, "best_val_ap": best_ap}, "rtf_encoder_clf.pt")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].plot(hist["tr_l"], label="train"); ax[0].plot(hist["va_l"], label="val"); ax[0].set_title("BCE"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(hist["tr_a"], label="train"); ax[1].plot(hist["va_a"], label="val"); ax[1].set_title("AUC"); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[2].plot(hist["tr_ap"], label="train"); ax[2].plot(hist["va_ap"], label="val"); ax[2].set_title("AP"); ax[2].legend(); ax[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 4. Evaluate + val-calibrated threshold


In [ ]:
@torch.no_grad()
def predict_probs(path):
    X, M, Y, subs, npts = load_rtf_tensors(path, stats)
    loader = DataLoader(DS(X, M, Y), batch_size=256, shuffle=False)
    model.eval()
    ps, ys = [], []
    for x, m, y in loader:
        logit = model(x.to(device), m.to(device))
        ps.append(torch.sigmoid(logit).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(ps), np.concatenate(ys).astype(int), subs, npts


def operating_points(y, p, name):
    fpr, tpr, thr = roc_curve(y, p)
    a = auc(fpr, tpr)
    ap = average_precision_score(y, p)
    print(f"\n{name}: N={len(p)} KN={(y==1).sum()} Imp={(y==0).sum()}")
    print(f"  KN mean P={p[y==1].mean():.4f}  Imp mean P={p[y==0].mean():.4f}")
    print(f"  ROC-AUC={a:.4f}  AP={ap:.4f}")
    chosen = {}
    for target in (0.90, 0.95):
        idxs = np.where(tpr >= target)[0]
        if len(idxs) == 0:
            print(f"  TPR>={target}: not reachable")
            continue
        i = idxs[0]
        chosen[target] = float(thr[i])
        cm = confusion_matrix(y, (p >= thr[i]).astype(int))
        print(f"  TPR>={target:.2f} → thr={thr[i]:.4f}  FPR={fpr[i]:.4f}  "
              f"TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")
    return a, ap, chosen


def plot_eval(y, p, name, a, ap):
    fpr, tpr, _ = roc_curve(y, p)
    prec, rec, _ = precision_recall_curve(y, p)
    kn, imp = y == 1, y == 0
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    axes[0].plot(fpr, tpr, label=f"AUC={a:.3f}"); axes[0].plot([0,1],[0,1],"k--",alpha=0.5)
    axes[0].set_title(f"ROC {name}"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(rec, prec, label=f"AP={ap:.3f}"); axes[1].set_title(f"PR {name}"); axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[2].hist(p[imp], bins=40, alpha=0.6, density=True, label="Imp")
    axes[2].hist(p[kn], bins=40, alpha=0.6, density=True, label="KN")
    axes[2].set_title(f"P(KN) {name}"); axes[2].legend(); axes[2].grid(alpha=0.3)
    plt.tight_layout(); plt.show()


vp, vy, vsub, vnpts = predict_probs(PARQUET_VAL)
va, vap, v_thr = operating_points(vy, vp, "Val")
plot_eval(vy, vp, "Val", va, vap)

tp, ty, tsub, tnpts = predict_probs(PARQUET_TEST)
ta, tap, _ = operating_points(ty, tp, "Test")
plot_eval(ty, tp, "Test", ta, tap)

if 0.95 in v_thr:
    thr95 = v_thr[0.95]
    pred = (tp >= thr95).astype(int)
    cm = confusion_matrix(ty, pred)
    print(f"\nTest @ val TPR>=0.95 thr={thr95:.4f}: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")
    fp = (pred == 1) & (ty == 0)
    print("FP by subclass:")
    for s, c in Counter(tsub[fp]).most_common(10):
        print(f"  {s:20s} {c}")

torch.save({
    "model": model.state_dict(), "stats": stats,
    "val_thr_tpr90": v_thr.get(0.90), "val_thr_tpr95": v_thr.get(0.95),
    "best_val_ap": best_ap, "test_auc": ta, "test_ap": tap,
}, "rtf_encoder_clf_calibrated.pt")
print("Saved rtf_encoder_clf_calibrated.pt")


## 5. Scalar logistic baseline (most important comparison)


In [ ]:
def scalars(path):
    df = pd.read_parquet(path)
    rows = []
    for _, g in df.groupby("objectId"):
        m, t = g["magpsf"].values, g["time_day"].values
        rows.append({
            "y": int(g["label_class"].iloc[0] == 1),
            "n": len(g),
            "dur": float(t.max() - t.min()) if len(t) > 1 else 0.0,
            "peak": float(m.min()),
            "mean_m": float(m.mean()),
            "std_m": float(m.std()) if len(m) > 1 else 0.0,
        })
    return pd.DataFrame(rows)

tr, te = scalars(PARQUET_TRAIN), scalars(PARQUET_TEST)
cols = ["n", "dur", "peak", "mean_m", "std_m"]
sc = StandardScaler().fit(tr[cols])
clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit(sc.transform(tr[cols]), tr["y"])
log_auc = roc_auc_score(te["y"], clf.predict_proba(sc.transform(te[cols]))[:, 1])
print(f"Logistic test AUC: {log_auc:.4f}")
print(f"RTF test AUC:      {ta:.4f}")
print("Weights:", dict(zip(cols, np.round(clf.coef_[0], 3))))
if ta - log_auc > 0.05:
    print("→ RTF adds meaningful sequence signal beyond scalars.")
elif abs(ta - log_auc) < 0.03:
    print("→ RTF ≈ logistic; value is mostly in hand features / data construction.")
else:
    print("→ RTF not clearly better than scalars on this split.")

rho, _ = spearmanr(tnpts, tp)
print(f"\nSpearman(n_points, P) ALL={rho:.3f}")
for lab, name in [(1, "KN"), (0, "Imp")]:
    m = ty == lab
    if m.sum() < 10:
        continue
    r, _ = spearmanr(tnpts[m], tp[m])
    print(f"  {name}: {r:.3f} (n={m.sum()})")
